# RAG for IoT Privacy Policies & Product Descriptions

A ready-to-run notebook to build a small Retrieval-Augmented Generation (RAG) system over **privacy policies** and **product descriptions** for popular smart-home devices using **local Ollama models**.

**What you'll get:**
- A mini corpus built from privacy policy pages, product pages, and (optionally) manuals
- Chunking + embeddings with an open model (`all-MiniLM-L6-v2`)
- A FAISS vector index for semantic retrieval
- A RAG pipeline that uses **Ollama** for local LLM generation (no API keys needed!)
- Utilities to add more devices via CSV

**Prerequisites:**
1. **Install Ollama**: Download from https://ollama.ai or run `brew install ollama`
2. **Start Ollama**: Run `ollama serve` in terminal
3. **Pull a model**: Run `ollama pull llama3` (or your preferred model)

**How to use:**
1. Run the setup cells (installs + imports).
2. Check Ollama status using the management utilities.
3. Inspect or edit the starter device list (URLs included).
4. Build the dataset and index.
5. Ask questions like _"What data does Ring Doorbell collect?"_

**Supported models:** llama3, llama3.1, mistral, phi3, codellama, and others available in Ollama.

> ⚠️ Scraping ethics: Respect robots.txt and terms of service. This demo pulls a few public pages. For production, implement rate limiting, error handling, and caching.

In [6]:
import json
import requests
import time
import re
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import urllib.parse
from typing import List, Dict, Any
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DeviceDatabaseGenerator:
    def __init__(self, vendor_mapping_file: str):
        with open(vendor_mapping_file, 'r') as f:
            self.vendor_mapping = json.load(f)
        
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        
        # DuckDuckGo search instance
        self.ddgs = DDGS()
        
        # Delay between searches to avoid rate limiting
        self.search_delay = 1
        
    def clean_device_name(self, device_name: str) -> str:
        """Clean device name for better search results"""
        # Remove common unnecessary words and characters
        cleaned = re.sub(r'\b(Home|Smart|WiFi|Wireless|IP|HD|4K)\b', '', device_name, flags=re.IGNORECASE)
        cleaned = re.sub(r'[^\w\s-]', '', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        return cleaned
    
    def search_urls(self, query: str, num_results: int = 5) -> List[str]:
        """Search DuckDuckGo and return URLs"""
        try:
            urls = []
            search_results = self.ddgs.text(query, max_results=num_results)
            
            for result in search_results:
                if result.get('href') and result['href'].startswith('http'):
                    urls.append(result['href'])
                    if len(urls) >= num_results:
                        break
            
            return urls
        except Exception as e:
            logger.error(f"Error searching for query '{query}': {str(e)}")
            return []
    
    def get_policy_urls(self, device: str, brand: str) -> List[str]:
        """Get privacy policy URLs for a device"""
        queries = [
            f"{brand} {device} privacy policy",
            f"{brand} privacy policy",
            f"{brand} {device} data collection policy",
            f"{brand} smart home privacy",
            f"{brand} IoT privacy policy",
            f"site:{self.get_brand_domain(brand)} privacy policy"
        ]
        
        all_urls = []
        for query in queries:
            try:
                urls = self.search_urls(query, 3)
                all_urls.extend(urls)
                time.sleep(self.search_delay)  # Be respectful to search engines
            except Exception as e:
                logger.warning(f"Error with query '{query}': {str(e)}")
                continue
        
        # Remove duplicates and filter relevant URLs
        unique_urls = list(dict.fromkeys(all_urls))
        policy_urls = [url for url in unique_urls if self.is_policy_url(url, brand)]
        
        return policy_urls[:5]
    
    def get_product_urls(self, device: str, brand: str) -> List[str]:
        """Get product page URLs for a device"""
        cleaned_device = self.clean_device_name(device)
        
        queries = [
            f"{brand} {cleaned_device} official product page",
            f"{brand} {cleaned_device} buy",
            f"{brand} {cleaned_device} store",
            f"site:{self.get_brand_domain(brand)} {cleaned_device}",
            f"{brand} {cleaned_device} specifications",
            f"{brand} {cleaned_device} product details"
        ]
        
        all_urls = []
        for query in queries:
            try:
                urls = self.search_urls(query, 3)
                all_urls.extend(urls)
                time.sleep(self.search_delay)
            except Exception as e:
                logger.warning(f"Error with query '{query}': {str(e)}")
                continue
        
        # Remove duplicates and filter relevant URLs
        unique_urls = list(dict.fromkeys(all_urls))
        product_urls = [url for url in unique_urls if self.is_product_url(url, brand, device)]
        
        return product_urls[:5]
    
    def get_manual_urls(self, device: str, brand: str) -> List[str]:
        """Get manual/support URLs for a device"""
        cleaned_device = self.clean_device_name(device)
        
        queries = [
            f"{brand} {cleaned_device} user manual",
            f"{brand} {cleaned_device} setup guide",
            f"{brand} {cleaned_device} support documentation",
            f"{brand} {cleaned_device} installation guide",
            f"site:{self.get_brand_domain(brand)} {cleaned_device} manual",
            f"{brand} {cleaned_device} help documentation"
        ]
        
        all_urls = []
        for query in queries:
            try:
                urls = self.search_urls(query, 3)
                all_urls.extend(urls)
                time.sleep(self.search_delay)
            except Exception as e:
                logger.warning(f"Error with query '{query}': {str(e)}")
                continue
        
        # Remove duplicates and filter relevant URLs
        unique_urls = list(dict.fromkeys(all_urls))
        manual_urls = [url for url in unique_urls if self.is_manual_url(url, brand)]
        
        return manual_urls[:5]
    
    def get_brand_domain(self, brand: str) -> str:
        """Get the primary domain for a brand"""
        domain_mapping = {
            'Amazon': 'amazon.com',
            'Google': 'google.com',
            'Apple': 'apple.com',
            'Samsung': 'samsung.com',
            'LG': 'lg.com',
            'Sony': 'sony.com',
            'Philips': 'philips.com',
            'Microsoft': 'microsoft.com',
            'Nintendo': 'nintendo.com',
            'Roku': 'roku.com',
            'Nest': 'nest.com',
            'Ring': 'ring.com',
            'Arlo': 'arlo.com',
            'TP-Link': 'tp-link.com',
            'Netgear': 'netgear.com',
            'Belkin': 'belkin.com',
            'Wyze': 'wyze.com',
            'Honeywell': 'honeywell.com',
            'Ecobee': 'ecobee.com',
            'LIFX': 'lifx.com',
            'Sonos': 'sonos.com',
            'Bose': 'bose.com',
            'iRobot': 'irobot.com',
            'Amcrest': 'amcrest.com',
            'Foscam': 'foscam.com',
            'Axis': 'axis.com',
            'iHome': 'ihome.com',
            'Brilliant': 'brilliant.tech',
            'Brother': 'brother.com',
            'Canary': 'canary.is',
            'Canon': 'canon.com',
            'Cisco': 'cisco.com',
            'Denon': 'denon.com',
            'Dyson': 'dyson.com',
            'HP': 'hp.com',
            'Logitech': 'logitech.com',
            'Leviton': 'leviton.com',
            'Lexmark': 'lexmark.com',
            'Libratone': 'libratone.com',
            'Lutron': 'lutron.com',
            'Miele': 'miele.com',
            'Netgear': 'netgear.com',
            'Nintendo': 'nintendo.com',
            'Nvidia': 'nvidia.com',
            'Onkyo': 'onkyo.com',
            'Ooma': 'ooma.com',
            'Osram': 'osram.com',
            'Polycom': 'polycom.com',
            'Epson': 'epson.com',
            'Reolink': 'reolink.com',
            'SiliconDust': 'silicondust.com',
            'SkyBell': 'skybell.com',
            'TiVo': 'tivo.com',
            'Vizio': 'vizio.com',
            'Nokia': 'nokia.com',
            'Xerox': 'xerox.com',
            'Xiaomi': 'mi.com',
            'Dahua': 'dahuasecurity.com',
            'Zmodo': 'zmodo.com',
            'LaMetric': 'lametric.com',
            'Meross': 'meross.com',
            'Blink': 'blinkforhome.com',
            'Shelly': 'shelly.cloud'
        }
        return domain_mapping.get(brand, f"{brand.lower().replace(' ', '').replace('-', '')}.com")
    
    def is_policy_url(self, url: str, brand: str) -> bool:
        """Check if URL is likely a privacy policy URL"""
        policy_keywords = ['privacy', 'policy', 'data-protection', 'terms', 'legal', 'gdpr']
        url_lower = url.lower()
        brand_domain = self.get_brand_domain(brand).lower()
        
        # Prefer URLs from the brand's official domain
        is_official = any(domain_part in url_lower for domain_part in brand_domain.split('.'))
        has_policy_keyword = any(keyword in url_lower for keyword in policy_keywords)
        
        # Exclude obviously irrelevant sites
        excluded_domains = ['reddit.com', 'youtube.com', 'twitter.com', 'facebook.com', 'instagram.com']
        is_excluded = any(domain in url_lower for domain in excluded_domains)
        
        return has_policy_keyword and (is_official or 'privacy' in url_lower) and not is_excluded
    
    def is_product_url(self, url: str, brand: str, device: str) -> bool:
        """Check if URL is likely a product page URL"""
        product_keywords = ['product', 'buy', 'store', 'shop', 'device', 'specifications', 'features']
        url_lower = url.lower()
        brand_domain = self.get_brand_domain(brand).lower()
        device_keywords = [word.lower() for word in device.split() if len(word) > 3]
        
        is_official = any(domain_part in url_lower for domain_part in brand_domain.split('.'))
        has_product_keyword = any(keyword in url_lower for keyword in product_keywords)
        has_device_keyword = any(keyword in url_lower for keyword in device_keywords)
        
        # Exclude obviously irrelevant sites
        excluded_domains = ['reddit.com', 'youtube.com', 'twitter.com', 'facebook.com', 'instagram.com', 'wikipedia.org']
        is_excluded = any(domain in url_lower for domain in excluded_domains)
        
        return (is_official or has_device_keyword) and not is_excluded
    
    def is_manual_url(self, url: str, brand: str) -> bool:
        """Check if URL is likely a manual/support URL"""
        manual_keywords = ['manual', 'support', 'help', 'guide', 'setup', 'documentation', 'install', 'instructions']
        url_lower = url.lower()
        brand_domain = self.get_brand_domain(brand).lower()
        
        is_official = any(domain_part in url_lower for domain_part in brand_domain.split('.'))
        has_manual_keyword = any(keyword in url_lower for keyword in manual_keywords)
        
        # Exclude obviously irrelevant sites
        excluded_domains = ['reddit.com', 'youtube.com', 'twitter.com', 'facebook.com', 'instagram.com']
        is_excluded = any(domain in url_lower for domain in excluded_domains)
        
        return has_manual_keyword and (is_official or 'support' in url_lower or 'manual' in url_lower) and not is_excluded
    
    def generate_device_entry(self, device: str, brand: str) -> Dict[str, Any]:
        """Generate a complete database entry for a device"""
        logger.info(f"Processing {brand} - {device}")
        
        try:
            policy_urls = self.get_policy_urls(device, brand)
            time.sleep(1)
            
            product_urls = self.get_product_urls(device, brand)
            time.sleep(1)
            
            manual_urls = self.get_manual_urls(device, brand)
            time.sleep(1)
            
            return {
                'device': device,
                'brand': brand,
                'policy_urls': policy_urls,
                'product_urls': product_urls,
                'manual_urls': manual_urls
            }
        
        except Exception as e:
            logger.error(f"Error processing {brand} - {device}: {str(e)}")
            return {
                'device': device,
                'brand': brand,
                'policy_urls': [],
                'product_urls': [],
                'manual_urls': [],
                'error': str(e)
            }
    
    def generate_complete_database(self, max_devices_per_brand: int = 3) -> List[Dict[str, Any]]:
        """Generate the complete device database"""
        database = []
        
        for brand, devices in self.vendor_mapping.items():
            logger.info(f"Processing brand: {brand}")
            
            # Limit devices per brand to avoid overwhelming the search
            devices_to_process = devices[:max_devices_per_brand]
            
            for device in devices_to_process:
                entry = self.generate_device_entry(device, brand)
                database.append(entry)
                
                # Add a delay between devices to be respectful
                time.sleep(2)
        
        return database
    
    def save_database(self, database: List[Dict[str, Any]], filename: str):
        """Save the database to a JSON file"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(database, f, indent=2, ensure_ascii=False)
        
        logger.info(f"Database saved to {filename}")

# Test function for a few devices
def test_search():
    """Test the search functionality with a few devices"""
    generator = DeviceDatabaseGenerator('rag_iot_data/device_database/vendor_device_mapping.json')
    
    test_devices = [
        ("Amazon Echo", "Amazon"),
        ("Google Home", "Google"),
        ("Apple TV", "Apple")
    ]
    
    for device, brand in test_devices:
        print(f"\n=== Testing {brand} {device} ===")
        entry = generator.generate_device_entry(device, brand)
        
        print(f"Policy URLs ({len(entry['policy_urls'])}): {entry['policy_urls']}")
        print(f"Product URLs ({len(entry['product_urls'])}): {entry['product_urls']}")
        print(f"Manual URLs ({len(entry['manual_urls'])}): {entry['manual_urls']}")

def main():
    generator = DeviceDatabaseGenerator('rag_iot_data/device_database/vendor_device_mapping.json')
    
    # Generate database for first 2 devices per brand (adjust as needed)
    database = generator.generate_complete_database(max_devices_per_brand=2)
    
    # Save the database
    generator.save_database(database, 'iot_device_database.json')
    
    # Print summary
    total_devices = len(database)
    devices_with_policies = len([d for d in database if d.get('policy_urls')])
    devices_with_products = len([d for d in database if d.get('product_urls')])
    devices_with_manuals = len([d for d in database if d.get('manual_urls')])
    
    print(f"\nDatabase Generation Summary:")
    print(f"Total devices processed: {total_devices}")
    print(f"Devices with policy URLs: {devices_with_policies}")
    print(f"Devices with product URLs: {devices_with_products}")
    print(f"Devices with manual URLs: {devices_with_manuals}")

# Uncomment to test with a few devices first
# test_search()

# Uncomment to run the full database generation
main()

/var/folders/b7/vk50rz2s03s3s6qqmwr8jmqm0000gn/T/ipykernel_5304/3302279133.py:26: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS()
2025-08-15 00:21:19,084 - INFO - Processing brand: Honeywell
2025-08-15 00:21:19,084 - INFO - Processing Honeywell - Honeywell Home Assistant
2025-08-15 00:21:19,360 - INFO - response: https://www.bing.com/search?q=Honeywell+Honeywell+Home+Assistant+privacy+policy 200
2025-08-15 00:21:21,672 - INFO - response: https://www.bing.com/search?q=Honeywell+privacy+policy 200
2025-08-15 00:21:24,040 - INFO - response: https://www.bing.com/search?q=Honeywell+Honeywell+Home+Assistant+data+collection+policy 200
2025-08-15 00:21:25,384 - INFO - response: https://www.bing.com/search?q=Honeywell+Honeywell+Home+Assistant+data+collection+policy&first=11&FORM=PERE 200
2025-08-15 00:21:27,539 - INFO - response: https://www.bing.com/search?q=Honeywell+smart+home+privacy 200
2025-08-15 00:21:29,


Database Generation Summary:
Total devices processed: 150
Devices with policy URLs: 150
Devices with product URLs: 150
Devices with manual URLs: 150


In [7]:
import json
import requests
import time
import re
import uuid
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from PyPDF2 import PdfReader
from io import BytesIO
from typing import List, Dict, Any
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
HEADERS = {'User-Agent': 'Mozilla/5.0 (RAG-Research-Demo)'}
MAX_BYTES = 2_000_000  # ~2MB per file
MAX_WORKERS = 10  # Number of threads
DATA_DIR = Path('rag_iot_data/device_database')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Thread-safe counter for progress tracking
class ThreadSafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()
    
    def increment(self):
        with self._lock:
            self._value += 1
            return self._value
    
    @property
    def value(self):
        with self._lock:
            return self._value

def fetch_url(url: str, timeout: int = 20) -> bytes:
    """Fetch URL content with error handling"""
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    if len(r.content) > MAX_BYTES:
        raise ValueError(f'Content too large: {len(r.content)} bytes > {MAX_BYTES}')
    return r.content

def html_to_text(content: bytes) -> str:
    """Convert HTML content to clean text"""
    soup = BeautifulSoup(content, 'html.parser')
    # Remove script/style/nav/footer/header elements
    for tag in soup(['script','style','nav','footer','header','noscript']):
        tag.decompose()
    
    # Extract text from relevant elements
    texts = [el.get_text(' ', strip=True) for el in soup.find_all(['p','li','h1','h2','h3','h4','h5','div','span'])]
    text = '\n'.join([t for t in texts if t and len(t.strip()) > 10])
    
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def pdf_to_text(content: bytes) -> str:
    """Convert PDF content to text"""
    try:
        reader = PdfReader(BytesIO(content))
        out = []
        for page in reader.pages:
            t = page.extract_text() or ''
            out.append(t)
        text = '\n'.join(out)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as e:
        logger.error(f"Error extracting PDF text: {e}")
        return ""

def smart_get_text(url: str) -> str:
    """Smart text extraction based on content type"""
    try:
        content = fetch_url(url)
        
        # Check if it's a PDF by URL extension or magic bytes
        if url.lower().endswith('.pdf') or (content[:4] == b'%PDF'):
            return pdf_to_text(content)
        else:
            return html_to_text(content)
    except Exception as e:
        logger.error(f"Error fetching {url}: {e}")
        return ""

def chunk_text(text: str, max_tokens: int = 350, overlap: int = 40) -> List[str]:
    """Split text into overlapping chunks"""
    if not text or len(text.strip()) < 50:
        return []
    
    # Simple whitespace tokenization approximation
    words = text.split()
    if len(words) <= max_tokens:
        return [text]
    
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i+max_tokens]
        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text)
        i += max(1, max_tokens - overlap)
        
        # Break if we're past the end
        if i >= len(words):
            break
    
    return chunks

def process_single_url(args) -> List[Dict[str, Any]]:
    """Process a single URL and return chunk records"""
    url, device, brand, kind, counter, total_urls = args
    
    try:
        text = smart_get_text(url)
        current_count = counter.increment()
        
        if not text or len(text) < 200:
            logger.warning(f"[{current_count}/{total_urls}] Too little text from {url} ({kind}) - {device}")
            return []
        
        chunks = chunk_text(text, max_tokens=350, overlap=40)
        
        if not chunks:
            logger.warning(f"[{current_count}/{total_urls}] No chunks created from {url} ({kind}) - {device}")
            return []
        
        records = []
        for idx, chunk in enumerate(chunks):
            records.append({
                'id': str(uuid.uuid4()),
                'device': device,
                'brand': brand,
                'kind': kind,
                'url': url,
                'chunk_id': idx,
                'text': chunk
            })
        
        logger.info(f"[{current_count}/{total_urls}] OK: {device} {kind} -> {url} [{len(chunks)} chunks]")
        return records
        
    except Exception as e:
        current_count = counter.increment()
        logger.error(f"[{current_count}/{total_urls}] ERR: {url} -> {e}")
        return []

def build_corpus_from_database(database_file: str = 'iot_device_database.json') -> pd.DataFrame:
    """Build corpus from device database using multithreading"""
    
    # Load the device database
    with open(database_file, 'r') as f:
        device_database = json.load(f)
    
    logger.info(f"Loaded {len(device_database)} devices from database")
    
    # Prepare all URL processing tasks
    all_tasks = []
    counter = ThreadSafeCounter()
    
    for entry in device_database:
        device = entry['device']
        brand = entry.get('brand', '')
        
        # Process each type of URL
        for kind, urls in [
            ('privacy_policy', entry.get('policy_urls', [])),
            ('product_page', entry.get('product_urls', [])),
            ('manual', entry.get('manual_urls', []))
        ]:
            for url in urls:
                if url and url.startswith('http'):  # Validate URL
                    all_tasks.append((url, device, brand, kind, counter, len([
                        u for e in device_database 
                        for urls_list in [e.get('policy_urls', []), e.get('product_urls', []), e.get('manual_urls', [])]
                        for u in urls_list if u and u.startswith('http')
                    ])))
    
    total_urls = len(all_tasks)
    logger.info(f"Processing {total_urls} URLs with {MAX_WORKERS} threads")
    
    # Process URLs in parallel
    all_records = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_task = {executor.submit(process_single_url, task): task for task in all_tasks}
        
        # Collect results as they complete
        for future in as_completed(future_to_task):
            try:
                records = future.result()
                all_records.extend(records)
            except Exception as e:
                task = future_to_task[future]
                logger.error(f"Task failed for {task[0]}: {e}")
    
    # Create DataFrame
    if all_records:
        df = pd.DataFrame(all_records)
        logger.info(f"Created corpus with {len(df)} chunks from {len(df['url'].unique())} unique URLs")
        
        # Print summary statistics
        print(f"\n=== Corpus Summary ===")
        print(f"Total chunks: {len(df)}")
        print(f"Unique devices: {df['device'].nunique()}")
        print(f"Unique brands: {df['brand'].nunique()}")
        print(f"Unique URLs: {df['url'].nunique()}")
        print(f"\nChunks by type:")
        print(df['kind'].value_counts())
        print(f"\nTop 10 devices by chunk count:")
        print(df['device'].value_counts().head(10))
        
        return df
    else:
        logger.warning("No records were created!")
        return pd.DataFrame()

def save_corpus(df: pd.DataFrame, filename: str = 'corpus.parquet'):
    """Save corpus to disk"""
    if df.empty:
        logger.warning("Cannot save empty corpus")
        return
    
    corpus_path = DATA_DIR / filename
    df.to_parquet(corpus_path, index=False)
    logger.info(f"Corpus saved to {corpus_path}")
    
    # Also save a sample as JSON for inspection
    sample_path = DATA_DIR / 'corpus_sample.json'
    sample_df = df.head(10)
    sample_df.to_json(sample_path, orient='records', indent=2)
    logger.info(f"Sample saved to {sample_path}")

# Main execution
def main():
    logger.info("Starting corpus building process...")
    
    # Build corpus from the generated database
    df = build_corpus_from_database('iot_device_database.json')
    
    if not df.empty:
        # Save the corpus
        save_corpus(df, 'device_data_corpus.parquet')

        # Display first few rows
        print(f"\n=== First 5 Records ===")
        print(df.head())
        
        return df
    else:
        logger.error("Failed to build corpus - no data collected")
        return None

# Run the main function
if __name__ == "__main__":
    corpus_df = main()

/Users/mdjakaria/ncsu-nyu/iot-inspector-client/.venv/lib/python3.13/site-packages/PyPDF2/__init__.py:21: DeprecationWarning: PyPDF2 is deprecated. Please move to the pypdf library instead.
  warnings.warn(
2025-08-15 19:23:08,735 - INFO - Starting corpus building process...
2025-08-15 19:23:08,736 - INFO - Loaded 150 devices from database
2025-08-15 19:23:08,897 - INFO - Processing 2130 URLs with 10 threads
2025-08-15 19:23:09,307 - INFO - [1/2130] OK: Honeywell Home Assistant product_page -> https://automation.honeywell.com/us/en/support/software-downloads [34 chunks]
2025-08-15 19:23:09,466 - ERROR - Error fetching https://www.honeywell.com/us/en/privacy-statement: Content too large: 2360545 bytes > 2000000
2025-08-15 19:23:09,468 - WARNING - [2/2130] Too little text from https://www.honeywell.com/us/en/privacy-statement (privacy_policy) - Honeywell Home Assistant
2025-08-15 19:23:09,631 - ERROR - Error fetching https://sps-support.honeywell.com/s/article/How-to-obtain-service-manual


=== Corpus Summary ===
Total chunks: 91611
Unique devices: 125
Unique brands: 127
Unique URLs: 1396

Chunks by type:
kind
privacy_policy    57192
manual            20613
product_page      13806
Name: count, dtype: int64

Top 10 devices by chunk count:
device
Frontier Silicon              5315
Google Chromecast             4826
Arcadyan HMB2260              4083
Xbox One                      3565
Microsoft Gaming Console      3232
Miele Smart Appliance         2304
LG TV                         2194
Chamberlain MyQ Smart Home    1513
Xerox Printer                 1392
Grandstream VoIP              1379
Name: count, dtype: int64


2025-08-15 19:29:17,203 - INFO - Corpus saved to rag_iot_data/device_database/device_data_corpus.parquet
2025-08-15 19:29:17,207 - INFO - Sample saved to rag_iot_data/device_database/corpus_sample.json



=== First 5 Records ===
                                     id                    device      brand  \
0  ca8004f6-30dc-4684-9e53-b2de04205ac5  Honeywell Home Assistant  Honeywell   
1  444924a3-0a80-4aba-bfed-41ea6729f448  Honeywell Home Assistant  Honeywell   
2  833603a9-1656-4a49-b4d0-173009cd774d  Honeywell Home Assistant  Honeywell   
3  9181cb3c-4b27-4b1e-a263-d8b760337e93  Honeywell Home Assistant  Honeywell   
4  9ca0dfbc-4a9a-496a-beb3-cc558fc050bc  Honeywell Home Assistant  Honeywell   

           kind                                                url  chunk_id  \
0  product_page  https://automation.honeywell.com/us/en/support...         0   
1  product_page  https://automation.honeywell.com/us/en/support...         1   
2  product_page  https://automation.honeywell.com/us/en/support...         2   
3  product_page  https://automation.honeywell.com/us/en/support...         3   
4  product_page  https://automation.honeywell.com/us/en/support...         4   

             

## Build Embeddings + FAISS Index
Creates a vector index and saves artifacts to disk for reuse in later sessions.

In [9]:
import json
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM = 384  # known dimension for the above model
TOP_K = 5

df = pd.read_parquet(DATA_DIR / 'device_data_corpus.parquet')


if len(df) == 0:
    raise RuntimeError('Corpus is empty. Check URLs or add more.')

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(df['text'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

faiss_path = DATA_DIR / 'index.faiss'
faiss.write_index(index, str(faiss_path))

emb_path = DATA_DIR / 'embeddings.npy'
np.save(emb_path, embeddings)

meta_path = DATA_DIR / 'metadata.json'
with open(meta_path, 'w') as f:
    json.dump(df.to_dict(orient='records'), f)

print('Saved index:', faiss_path)
print('Saved embeddings:', emb_path)
print('Saved metadata:', meta_path)

2025-08-15 19:44:45,176 - INFO - Loading faiss.
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
2025-08-15 19:44:45,270 - INFO - Successfully loaded faiss.
2025-08-15 19:44:50,793 - INFO - Use pytorch device_name: mps
2025-08-15 19:44:50,794 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1432 [00:00<?, ?it/s]

Saved index: rag_iot_data/device_database/index.faiss
Saved embeddings: rag_iot_data/device_database/embeddings.npy
Saved metadata: rag_iot_data/device_database/metadata.json


## Ollama Setup & Management
Quick utilities to check Ollama status, list models, and pull new models if needed.

In [10]:
import requests
import os

# Ollama Configuration for local LLM
class OllamaConfig:
    def __init__(self):
        self.ollama_url = os.getenv('OLLAMA_URL', 'http://localhost:11434')
        self.model_name = os.getenv('LLM_MODEL', 'llama3:latest')
        self.timeout = int(os.getenv('LLM_TIMEOUT', '30'))
        self.max_retries = int(os.getenv('LLM_MAX_RETRIES', '3'))
    
    
    def is_available(self) -> bool:
        """Check if Ollama is running and accessible"""
        try:
            response = requests.get(f"{self.ollama_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def get_available_models(self) -> List[str]:
        """Get list of available models from Ollama"""
        try:
            response = requests.get(f"{self.ollama_url}/api/tags", timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                return [model['name'] for model in models]
        except:
            return []
        return []

# Initialize Ollama config
ollama_config = OllamaConfig()

def check_ollama_status():
    """Check Ollama status and available models"""
    if ollama_config.is_available():
        models = ollama_config.get_available_models()
        print(f"✅ Ollama is running at {ollama_config.ollama_url}")
        print(f"📋 Available models: {', '.join(models) if models else 'None'}")
        
        if ollama_config.model_name in models:
            print(f"🎯 Current model '{ollama_config.model_name}' is available")
        else:
            print(f"⚠️ Current model '{ollama_config.model_name}' not found")
            if models:
                print(f"💡 You can use: {models[0]}")
    else:
        print(f"❌ Ollama not available at {ollama_config.ollama_url}")
        print("💡 Start Ollama with: ollama serve")
        print(f"💡 Pull model with: ollama pull {ollama_config.model_name}")

def pull_ollama_model(model_name: str = None):
    """Pull a model in Ollama"""
    model_name = model_name or ollama_config.model_name
    try:
        url = f"{ollama_config.ollama_url}/api/pull"
        payload = {"name": model_name}
        print(f"⬇️ Pulling model '{model_name}'... (this may take a while)")
        
        response = requests.post(url, json=payload, timeout=300)
        if response.status_code == 200:
            print(f"✅ Model '{model_name}' pulled successfully")
            return True
        else:
            print(f"❌ Failed to pull model: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error pulling model: {e}")
        return False

def test_ollama_model(model_name: str = None):
    """Test if a model is working"""
    model_name = model_name or ollama_config.model_name
    test_prompt = "Respond with exactly: 'Hello, I am working correctly!'"
    
    print(f"🧪 Testing model '{model_name}'...")
    response = call_ollama(test_prompt)
    
    if "❌" not in response:
        print(f"✅ Model '{model_name}' is working!")
        print(f"📝 Response: {response}")
        return True
    else:
        print(f"❌ Model test failed: {response}")
        return False


def call_ollama(prompt: str, system_prompt: str = None) -> str:
    """Call Ollama API with proper error handling"""
    
    if not OLLAMA_AVAILABLE:
        return "❌ Ollama is not available. Please start Ollama and ensure the model is installed."
    
    for attempt in range(ollama_config.max_retries):
        try:
            if system_prompt:
                # Use chat API format
                url = f"{ollama_config.ollama_url}/api/chat"
                payload = {
                    "model": ollama_config.model_name,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": prompt}
                    ],
                    "stream": False,
                    "options": {
                        "temperature": 0.2,
                        "top_p": 0.9,
                        "num_predict": 600
                    }
                }
            else:
                # Use generate API format
                url = f"{ollama_config.ollama_url}/api/generate"
                payload = {
                    "model": ollama_config.model_name,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.2,
                        "top_p": 0.9,
                        "num_predict": 600
                    }
                }
            
            response = requests.post(url, json=payload, timeout=ollama_config.timeout)
            response.raise_for_status()
            
            result = response.json()
            
            if system_prompt:
                return result.get('message', {}).get('content', 'No response').strip()
            else:
                return result.get('response', 'No response').strip()
                
        except requests.exceptions.ConnectionError:
            if attempt == ollama_config.max_retries - 1:
                return f"❌ Could not connect to Ollama at {ollama_config.ollama_url}"
        except requests.exceptions.Timeout:
            if attempt == ollama_config.max_retries - 1:
                return "❌ Request timed out"
        except Exception as e:
            if attempt == ollama_config.max_retries - 1:
                return f"❌ Error: {str(e)}"
    
    return "❌ All retry attempts failed"

# Run initial status check
print("=== Ollama Status Check ===")
# check_ollama_status()

# Uncomment to pull a specific model
# pull_ollama_model("llama3")

# Uncomment to test the current model
# test_ollama_model()

# Check Ollama availability
OLLAMA_AVAILABLE = ollama_config.is_available()
if OLLAMA_AVAILABLE:
    available_models = ollama_config.get_available_models()
    print(f"✅ Ollama is running at {ollama_config.ollama_url}")
    print(f"📋 Available models: {', '.join(available_models) if available_models else 'None'}")
    if ollama_config.model_name in available_models:
        print(f"🎯 Using model: {ollama_config.model_name}")

    else:
        print(f"⚠️ Model '{ollama_config.model_name}' not found. Available: {available_models}")
else:
    print(f"❌ Ollama not available at {ollama_config.ollama_url}")

    print("💡 Start Ollama with: ollama serve")
    print(f"💡 Pull model with: ollama pull {ollama_config.model_name}")


=== Ollama Status Check ===
✅ Ollama is running at http://localhost:11434
📋 Available models: llama3:latest, phi3:latest
🎯 Using model: llama3:latest


## Retriever & (Optional) Generator
We implement a basic retriever over FAISS and an optional generator using OpenAI if `OPENAI_API_KEY` is set. If no key is set, you'll still get the top passages with citations.

In [12]:
def load_artifacts():
    df = pd.read_parquet(DATA_DIR / 'device_data_corpus.parquet')
    index = faiss.read_index(str(DATA_DIR / 'index.faiss'))
    embeddings = np.load(DATA_DIR / 'embeddings.npy')
    return df, index, embeddings

def retrieve(query: str, k: int = TOP_K) -> pd.DataFrame:
    model = SentenceTransformer(MODEL_NAME)
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q_emb, k)
    hits = df.iloc[I[0]].copy()
    hits['score'] = D[0]
    return hits[['score','device','brand','kind','url','chunk_id','text']]


def synthesize_answer(query: str, hits: pd.DataFrame) -> str:
    context_blocks = []
    for _, row in hits.iterrows():
        cite = f"[{row['device']} | {row['kind']} | {row['url']}]"
        context_blocks.append(f"SOURCE: {cite}\n{row['text']}")
    context_text = '\n\n'.join(context_blocks)

    if OLLAMA_AVAILABLE:
        system_prompt = (
            "You are a careful assistant analyzing IoT device privacy policies and product information. "
            "Answer only using the provided sources. Cite sources inline after each relevant claim using "
            "[device | kind | url] format. If information is missing or unclear, explicitly state so. "
            "Be concise but thorough in your analysis."
        )
        
        user_prompt = f"""Question: {query}

Sources:
{context_text}

Please analyze the sources above to answer the question. Include inline citations and be specific about what each device does regarding the queried topic."""

        try:
            response = call_ollama(user_prompt, system_prompt)
            if response.startswith("❌"):
                return f"{response}\n\n📚 Top passages for reference:\n\n{context_text}"
            return response
        except Exception as e:
            return f"❌ Generation error: {e}\n\n📚 Top passages:\n\n{context_text}"
    else:
        # No Ollama: return the top passages as the 'answer'
        header = f"❌ Ollama not available at {ollama_config.ollama_url}. Showing top retrieved passages with citations.\n\n"
        return header + context_text

# Load artifacts for retrieval
df, index, embeddings = load_artifacts()
print(f'✅ Artifacts loaded. Total chunks: {len(df)}')

✅ Artifacts loaded. Total chunks: 91611


## Ask Questions
Try queries like:
- *What data does Ring Video Doorbell collect?*
- *Does Nest Thermostat share data with third parties?*
- *What features does Philips Hue include?*

**Note:** This uses Ollama for local LLM generation. If Ollama is not available, you'll get the retrieved passages with citations instead.

**Troubleshooting:**
- If you see "❌ Ollama not available", run `ollama serve` in terminal
- If model is missing, run `ollama pull llama3` (or your preferred model)
- Use the Ollama management utilities above to check status and pull models

In [13]:
query = "What data does Ring Video Doorbell collect?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

2025-08-15 20:36:25,641 - INFO - Use pytorch device_name: mps
2025-08-15 20:36:25,641 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,device,brand,kind,url,chunk_id,text
35057,0.682913,Ring Security Camera,Ring,product_page,https://ring.com/collections/home-security-cam...,38,"you have a subscription to Ring Home, your Rin..."
79336,0.673129,Ring Stick Up Camera Elite,Misc,product_page,https://ring.com/products/stick-up-security-ca...,34,until after your trial ends. If you have a sub...
34707,0.671096,Ring Security Camera,Ring,privacy_policy,https://ring.com/privacy,25,and mobile apps. Web and app analytics service...
35059,0.660474,Ring Security Camera,Ring,product_page,https://ring.com/collections/home-security-cam...,40,more about Ring Home. If you have a subscripti...
89076,0.658867,Smanos Doorbell,Smanos,product_page,https://manualmachine.com/smanos/smartvideodoo...,30,Manual Welcome to the Smart Home Family! Thank...



---

According to the provided sources, Ring Video Doorbell collects various types of data. Here's a breakdown:

* Motion event videos: If you have a subscription to Ring Home, your Ring motion event videos are stored in the cloud for up to 180 days for doorbells and home cameras [1]. Without a subscription, you can view real-time video but your videos will not be saved.
* Photos: Photos captured by Ring devices will be saved to your Ring account for up to 7 days, so you can review them at any time [1].
* Video recordings: If you have a Ring Home Plan (including a free trial), Ring records and stores videos when motion is detected by your device, the video doorbell button is pressed, or you initiate Live View. Stored recordings are automatically deleted following the expiration of your Ring Home Plan's applicable storage period [2].

It's worth noting that if you create a share link for one of your Ring videos, a copy of that video will be made in the cloud to facilitate sharing [2]. 

In [14]:
query = "Does Nest Thermostat share data with third parties?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

2025-08-17 21:28:27,438 - INFO - Use pytorch device_name: mps
2025-08-17 21:28:27,439 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,device,brand,kind,url,chunk_id,text
25910,0.719014,Ecobee Smart WiFi Thermostat,Ecobee,privacy_policy,https://www.cnet.com/home/smart-home/smart-the...,45,a closer look. Do smart thermostats share your...
25942,0.714349,Ecobee Smart WiFi Thermostat,Ecobee,privacy_policy,https://www.cnet.com/home/smart-home/smart-the...,77,closer look. Do smart thermostats share your d...
25947,0.709993,Ecobee Smart WiFi Thermostat,Ecobee,privacy_policy,https://www.cnet.com/home/smart-home/smart-the...,82,to monitor specific temperatures. Thermostat s...
25916,0.706914,Ecobee Smart WiFi Thermostat,Ecobee,privacy_policy,https://www.cnet.com/home/smart-home/smart-the...,51,can go anywhere to monitor specific temperatur...
1796,0.702589,Honeywell Smart Thermostat,Honeywell,privacy_policy,https://wonchecd.com/smart-thermostats-and-you...,16,parties? Thermostat sensors can go anywhere to...



---

According to the provided sources, Nest Thermostat may share data with third parties depending on their partnerships or deals [1]. However, some brands like Resideo require explicit customer permission to share data with their third-party connections, which also helps meet privacy regulations in places like California or the EU [1].

Google Nest has a similar rule for its data sharing program but has a notable exception for connected services, saying, "We will ask your permission before sharing your personal information with third parties for purposes other than at your request or to provide Nest's Products, and to do so only when we think they will provide you with a welcome additional service" [1].

Ecobee also seems to require consent, although their language is vaguer. Brands like Sensi outright say they won't sell personal information to third parties [1].

In summary, Nest Thermostat may share data with third parties depending on their partnerships or deals, while Ecobee re

In [15]:
query = "What features does Philips Hue include?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

2025-08-17 21:31:51,485 - INFO - Use pytorch device_name: mps
2025-08-17 21:31:51,485 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,device,brand,kind,url,chunk_id,text
11785,0.662563,Philips Hue Bridge,Philips,product_page,https://hueblog.com/2025/08/14/hue-bridge-pro-...,33,July 2021 - 7:30 2023 Ambilight TVs no longer ...
11768,0.658147,Philips Hue Bridge,Philips,product_page,https://hueblog.com/2025/08/14/hue-bridge-pro-...,16,September An overview of the rumors 12. August...
11760,0.654429,Philips Hue Bridge,Philips,product_page,https://hueblog.com/2025/08/14/hue-bridge-pro-...,8,Plenty of PoE /USB-C splitters out there too →...
11776,0.648927,Philips Hue Bridge,Philips,product_page,https://hueblog.com/2025/08/14/hue-bridge-pro-...,24,getting an upgrade. This is intended to ensure...
11756,0.648679,Philips Hue Bridge,Philips,product_page,https://hueblog.com/2025/08/14/hue-bridge-pro-...,4,March 2023 - 8:00 Philips Hue 4.1: Group lamps...



---

According to the provided sources, Philips Hue includes features such as:

* Group lamps and control them together (Philips Hue 4.1: Group lamps and control them together [Philips Hue Bridge | product_page | https://hueblog.com/2025/08/14/hue-bridge-pro-and-other-new-products-appear-on-the-hue-website/])
* Brighter bulbs with a difference of 40% less energy consumption and ultra-deep dimming down to 0.2% (Brighter Philips Hue bulbs: How big is the difference? [Philips Hue Bridge | product_page | https://hueblog.com/2025/08/14/hue-bridge-pro-and-other-new-products-appear-on-the-hue-website/])
* MotionAware technology, which allows smart lights to be turned into motion sensors (Hue Bridge Pro: USB-C, Wi-Fi, and much more power [Philips Hue Bridge | product_page | https://hueblog.com/2025/08/14/hue-bridge-pro-and-other-new-products-appear-on-the-hue-website/])
* Sync TV app for a seamless viewing experience (The important answers about the Philips Hue Sync TV app [Philips Hue Bridge